In [ ]:
from skimage.io import imread
import stackview
import pyclesperanto as cle  # version 0.14.2
import napari_segment_blobs_and_things_with_membranes as nsbatwm  # version 0.3.12
import napari_simpleitk_image_processing as nsitk  # version 0.4.9
import napari
viewer = napari.Viewer() #  啟動napari視窗

## Activate napari assistant (optional)

In [ ]:
viewer.window.add_plugin_dock_widget(plugin_name='napari-assistant', widget_name='Assistant')

## 1. Open image
There are two common ways to get an image into the workflow. Choose one of the following methods.

#### Method.1 Open Sample
Use a built-in dataset from napari so that everyone works with the same image
viewer.open_sample('napari', 'cells3d')

In [ ]:
viewer.open_sample('napari', 'cells3d')

#### Method.2 Read image data from file (for your own data)
Use this method when analyzing your own images or running batch processing.

In [ ]:
from pathlib import Path # 使用 pathlib 處理相對路徑

# Example 1: 同一層路徑 file in the same folder
# filename = Path("cells3d.tif")

# Example 2: 子資料夾路徑 file in a subfolder 
filename = Path('data') / "cells3d.tif"

# Example 3: 絕對路徑 full path (Windows System)
# filename = Path("C:/Users/yourname/Desktop/EABIAS python/LESSON4/data/cells3d.tif")

# 執行讀取 
image = imread(filename) 
print (image.shape)


# Visualizing images: add image to viewer 加入 Viewer

viewer.add_image(
    image,
    channel_axis=1,
    name=['membrane', 'nuclei'],
    colormap=['magenta', 'green']
)

## 2. Select nuclei channel

In [ ]:
image_nuclei = viewer.layers['nuclei'].data

## 3. gaussian blur

In [ ]:
image1_G = nsitk.gaussian_blur(image_nuclei, 6.0, 6.0, 2.0)
viewer.add_image(image1_G, name='Result of Gaussian (n-SimpleITK)')
napari.utils.nbscreenshot(viewer)
#napari.utils.nbscreenshot(viewer, canvas_only=True)

## 4. threshold otsu

In [ ]:
image2_T = nsitk.threshold_otsu(image1_G)
viewer.add_labels(
    image2_T, name='Result of Threshold (Otsu et al 1979, n-SimpleITK)')
napari.utils.nbscreenshot(viewer, canvas_only=True)

## 5. binary fill holes

In [ ]:
image3_B = nsitk.binary_fill_holes(image2_T)
viewer.add_labels(image3_B, name='Result of Binary fill holes (n-SimpleITK)')
#for layer in viewer.layers[:-1]:
#    layer.visible = False
napari.utils.nbscreenshot(viewer, canvas_only=True)

## 6. split touching objects

In [ ]:
image4_S = nsbatwm.split_touching_objects(image3_B, 6.0)
viewer.add_labels(image4_S, name='Result of Split touching objects (nsbatwm)')
for layer in viewer.layers[:-1]:
    layer.visible = False
napari.utils.nbscreenshot(viewer, canvas_only=True)

## 7. connected component labeling

In [ ]:
image5_C = nsitk.connected_component_labeling(image4_S)
viewer.add_labels(
    image5_C, name='Result of Connected component labeling (n-SimpleITK)')
for layer in viewer.layers[:-1]:
    layer.visible = False
napari.utils.nbscreenshot(viewer, canvas_only=True)

## 8. exclude labels on edges

In [ ]:
image6_eloe = cle.exclude_labels_on_edges(image5_C, None, True, True, True)
viewer.add_labels(
    image6_eloe, name='Result of exclude_labels_on_edges (pyclesperanto)')
for layer in viewer.layers[:-1]:
    layer.visible = False
napari.utils.nbscreenshot(viewer, canvas_only=True)

## 9.mean intensity map

In [ ]:
image7_mim = cle.mean_intensity_map(image_nuclei, image6_eloe)
viewer.add_image(
    image7_mim, name='Result of mean_intensity_map (pyclesperanto)')
for layer in viewer.layers[:-1]:
    layer.visible = False
viewer.layers[-1].colormap = 'inferno'
viewer.layers[-1].reset_contrast_limits()
napari.utils.nbscreenshot(viewer, canvas_only=True)

## Table 

In [ ]:
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np

label_img = np.asarray(image6_eloe)
intensity_img = np.asarray(image_nuclei)

statistics = regionprops_table(
    label_img,                    # label image
    intensity_image=intensity_img,   # 原始影像
    properties=[
        'label',
        'area',
        'mean_intensity',
        'centroid',       #position
#        'eccentricity'    #shape
    ]
)

statistics_df = pd.DataFrame(statistics)
statistics_df.head()

## Export to csv

In [ ]:
statistics_df.to_csv("results.csv", index=False, encoding="utf_8_sig")
print("Results saved to results.csv")